# 06 · Differential response with replicate-aware statistics
Cells from one perturbation are **not independent experiments** (shared batch, shared guide, shared library prep). Treating each cell as a replicate inflates significance. We aggregate to **pseudobulk replicates (batches)** and use Welch's t-test with Benjamini–Hochberg FDR, then contrast with a naive cell-level Wilcoxon test.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import scanpy as sc
from perturbai.features.de import pseudobulk_de
adata = sc.read_h5ad(P.processed / artifact("subset_adata", TAG))
delta = pd.read_parquet(P.processed / artifact("effects", TAG))
labels = adata.obs[PERT].astype(str).values
batch = adata.obs[BATCH].astype(str).values if BATCH in adata.obs else None
norm = np.linalg.norm(delta.values, axis=1)
chosen = list(delta.index[np.argsort(-norm)[:10]]) + list(delta.index[np.argsort(norm)[:3]])
rows, tables = [], {}
for p in chosen:
    try:
        de = pseudobulk_de(adata.X, labels, adata.var_names, p, CTRL, batch, min_cells_rep=5 if not SYNTHETIC else 3)
    except ValueError as e:
        print("skip", p, e); continue
    tables[p] = de
    sig = de[(de.padj < 0.05) & (de.log_fc.abs() > 0.1)]
    rows.append({"perturbation": p, "n_sig": len(sig), "n_down": int((sig.log_fc < 0).sum()),
                 "self_log_fc": de.loc[p, "log_fc"] if p in de.index else np.nan,
                 "self_padj": de.loc[p, "padj"] if p in de.index else np.nan})
summary = pd.DataFrame(rows); summary.to_csv(P.results / f"{TAG}_de_summary.csv", index=False); summary

In [ ]:
p0 = summary.sort_values("n_sig", ascending=False).perturbation.iloc[0]
de = tables[p0]
plt.scatter(de.log_fc, -np.log10(de.pval + 1e-300), s=6, c=(de.padj < 0.05), cmap="coolwarm")
plt.xlabel("log fold-change (log-norm)"); plt.ylabel("-log10 p"); plt.title(f"{p0}: pseudobulk DE (replicates = {int(de.n_rep_pert.iloc[0])})"); plt.show()
de.head(15)

## Why pseudobulk? Compare with cell-level Wilcoxon

In [ ]:
sub = adata[adata.obs[PERT].astype(str).isin([p0, CTRL])].copy()
sub.obs[PERT] = sub.obs[PERT].astype(str).astype("category")
sc.tl.rank_genes_groups(sub, PERT, groups=[p0], reference=CTRL, method="wilcoxon")
w = sc.get.rank_genes_groups_df(sub, group=p0)
print("cell-level Wilcoxon, FDR<0.05:", int((w.pvals_adj < 0.05).sum()),
      "| pseudobulk, FDR<0.05:", int((de.padj < 0.05).sum()))
# expect cell-level to call many more genes: cells are pseudo-replicates, so p-values are optimistic

**Interpretation:** is the knocked-down gene itself recovered (`self_log_fc` negative, small `self_padj`)? How many more genes does the naive test call, and why?